In [1]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
from GBM import simulate_GBM, simulate_GBM_antithetic, simulate_GBM_numba_parallel
from european import european_mc, european_mc_antithetic
import matplotlib.pyplot as plt
from scipy.stats import qmc

In [2]:
def american_longstaff_schwartz(S0, K, r, sigma, T, n, n_paths, type="put"):
    t, S = simulate_GBM(S0, r, sigma, T, n, n_paths)
    dt = T / n
    
    if type == "put":
        payoff_fn = lambda s: np.maximum(K - s, 0)
    else:
        payoff_fn = lambda s: np.maximum(s - K, 0)
    
    # cash_flows[i] = the discounted-to-its-own-exercise-time cash flow for path i
    # exercise_time[i] = which timestep path i was exercised at (or n_steps if never)
    cash_flows = payoff_fn(S[:, -1])  # start by assuming exercise at expiry
    exercise_time = np.full(n_paths, n)
    
    # walk backward, skip the last column (already handled above) and t=0 (nothing to decide before start)
    for step in range(n - 1, 0, -1):
        S_t = S[:, step]
        exercise_value = payoff_fn(S_t)
        itm = exercise_value > 0
        
        if itm.sum() == 0:
            continue
        
        # discount the recorded cash flow back to THIS timestep for the regression target
        time_to_cashflow = (exercise_time[itm] - step) * dt
        Y = cash_flows[itm] * np.exp(-r * time_to_cashflow)
        X = S_t[itm]
        
        # regress Y on polynomial basis of X
        A = np.vstack([np.ones_like(X), X, X**2]).T
        coeffs, _, _, _ = np.linalg.lstsq(A, Y, rcond=None)
        continuation_value = A @ coeffs
        
        # decide: exercise now if it beats the estimated continuation value
        exercise_now = exercise_value[itm] > continuation_value
        
        itm_indices = np.where(itm)[0]
        exercise_indices = itm_indices[exercise_now]
        
        cash_flows[exercise_indices] = exercise_value[itm][exercise_now]
        exercise_time[exercise_indices] = step
    
    # discount every path's cash flow back to time 0 from its own exercise time
    discounted = cash_flows * np.exp(-r * exercise_time * dt)
    price = discounted.mean()
    se = discounted.std(ddof=1) / np.sqrt(n_paths)
    return price, se

In [3]:
def american_binomial(S0, K, r, sigma, T, n, type="put"):
    dt = T / n
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    
    # terminal stock prices at each node, step n_steps
    j = np.arange(n + 1)
    S_terminal = S0 * (u ** (n - j)) * (d ** j)
    
    if type == "put":
        values = np.maximum(K - S_terminal, 0)
    else:
        #values = np.maximum(K - S_terminal, 0)  
        values = np.maximum(S_terminal - K, 0)
    
    # walk backward
    for step in range(n - 1, -1, -1):
        j = np.arange(step + 1)
        S_t = S0 * (u ** (step - j)) * (d ** j)
        
        continuation = discount * (p * values[:step+1] + (1-p) * values[1:step+2])
        
        if type == "put":
            exercise = np.maximum(K - S_t, 0)
        else:
            exercise = np.maximum(S_t - K, 0)
        
        values = np.maximum(continuation, exercise)
    
    return values[0]

In [7]:
def fast_poly_regression(X, Y, degree=2):
    A = np.vstack([X**i for i in range(degree+1)]).T
    coeffs = np.linalg.solve(A.T @ A, A.T @ Y)
    return A @ coeffs

In [8]:
def american_longstaff_schwartz_fast(S0, K, r, sigma, T, n, n_paths, type="put"):
    S = simulate_GBM_numba_parallel(S0, r, sigma, T, n, n_paths)  # <-- swap 1: Numba parallel sim
    dt = T / n
    
    if type == "put":
        payoff_fn = lambda s: np.maximum(K - s, 0)
    else:
        payoff_fn = lambda s: np.maximum(s - K, 0)
    
    cash_flows = payoff_fn(S[:, -1])
    exercise_time = np.full(n_paths, n)
    
    for step in range(n - 1, 0, -1):
        S_t = S[:, step]
        exercise_value = payoff_fn(S_t)
        itm = exercise_value > 0
        
        if itm.sum() == 0:
            continue
        
        time_to_cashflow = (exercise_time[itm] - step) * dt
        Y = cash_flows[itm] * np.exp(-r * time_to_cashflow)
        X = S_t[itm]
        
        continuation_value = fast_poly_regression(X, Y, degree=2)  
        
        exercise_now = exercise_value[itm] > continuation_value
        itm_indices = np.where(itm)[0]
        exercise_indices = itm_indices[exercise_now]
        
        cash_flows[exercise_indices] = exercise_value[itm][exercise_now]
        exercise_time[exercise_indices] = step
    
    discounted = cash_flows * np.exp(-r * exercise_time * dt)
    price = discounted.mean()
    se = discounted.std(ddof=1) / np.sqrt(n_paths)
    return price, se